# LIBERO eval — **중간 결과 스냅샷** (팀 공유용)

eval 이 **도는 중**에 실행 → 지금까지 끝난 것만 모아 진행률 격자 · SR · 떨림 · zip 을 만든다.
언제 돌려도 안전(읽기만 함). 다시 돌리면 그 시점 스냅샷을 새로 만든다(파일명에 시각 태그).

- **대상 = 4모델 × 4 seed = 16 eval**. 격자에서 완료/진행중/대기가 한눈에.
- 완료된 것만 SR mean±std, action(.pt) 있는 것만 jerk/LDJ/SPARC/SignFlip.
- 결과 zip 하나 = `outputs/final/share/libero_10_progress_<시각>.zip` → 팀에 이 파일만.


In [ ]:
import sys, json, csv, time
from pathlib import Path
from collections import defaultdict
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
import numpy as np

# ── 이번 LIBERO eval 대상 (진행률 = 이 격자 기준) ──
MODELS = ['acm', 'act', 'bimamba', 'bimamba_s7']
SEEDS  = [0, 1, 2, 3]
TASK   = 'libero_10'
TARGET_EP = cf.EVAL_N_EP                 # 500
EXPECTED = [(m, s) for m in MODELS for s in SEEDS]   # 16

ROOT = cf.OUTPUT_BASE / 'eval_clean' / TASK
OUT  = cf.OUTPUT_BASE / 'share' / f'{TASK}_progress'
OUT.mkdir(parents=True, exist_ok=True)
STAMP = time.strftime('%Y%m%d_%H%M')
print('scan:', ROOT, '| exists:', ROOT.is_dir())
print('대상:', len(EXPECTED), 'eval (', len(MODELS), '모델 ×', len(SEEDS), 'seed )  |  스냅샷:', STAMP)

## 1) 스캔 — 완료된 eval 찾기 (부분 완료 OK)


In [ ]:
# ── eval_info.json 이 있는 run 전부 스캔 (부분 완료 OK, 레이아웃 무관: seedN/ 든 seedN/rep0/ 든) ──
def parse_info(p):
    try:
        d = json.loads(p.read_text())
    except Exception:
        return None
    ov = d.get('overall', d)
    sr = ov.get('pc_success')
    if sr is None and 'success' in ov:
        sr = 100.0 * float(np.mean(ov['success']))
    return {'sr': sr, 'n_ep': ov.get('n_episodes')}

done = {}                       # (model, seed) -> {'sr':, 'n_ep':, 'path':, 'has_actions':}
if ROOT.is_dir():
    for info in sorted(ROOT.rglob('eval_info.json')):
        parts = info.parent.relative_to(ROOT).parts
        model = parts[0] if parts else '?'
        seed = next((int(p[4:]) for p in parts if p.startswith('seed') and p[4:].isdigit()), None)
        m = parse_info(info)
        if m is None or seed is None:
            continue
        done[(model, seed)] = {'sr': m['sr'], 'n_ep': m['n_ep'],
                               'path': str(info.parent.relative_to(ROOT)),
                               'has_actions': (info.parent / 'actions').is_dir()}

# 시작은 됐는데 eval_info 아직 없는 = 진행중
running = set()
for m, s in EXPECTED:
    d = cf.eval_clean_dir(m, s, TASK) if (m in cf.v23.MODEL_DIR_NAMES) else ROOT / m / f'seed{s}'
    if (m, s) not in done and Path(d).is_dir():
        running.add((m, s))
n_done = len(done)
print(f'완료 {n_done}/{len(EXPECTED)}   진행중(추정) {len(running)}   대기 {len(EXPECTED)-n_done-len(running)}')

## 2) 진행률 격자 (model × seed)


In [ ]:
# ── 진행률 격자: 행=모델, 열=seed. 값 = SR%(완료) / … (진행중) / · (대기) ──
def cell(m, s):
    if (m, s) in done:
        sr = done[(m, s)]['sr']
        return f'{sr:.1f}' if sr is not None else 'done?'
    return '…' if (m, s) in running else '·'

hdr = f"{'model':<14}" + ''.join(f'{("seed" + str(s)):>9}' for s in SEEDS) + f"{'done':>7}"
print(hdr); print('-' * len(hdr))
grid_rows = []
for m in MODELS:
    vals = [cell(m, s) for s in SEEDS]
    nd = sum((m, s) in done for s in SEEDS)
    print(f'{m:<14}' + ''.join(f'{v:>9}' for v in vals) + f'{nd}/{len(SEEDS):>3}')
    grid_rows.append({'model': m, **{f'seed{s}': cell(m, s) for s in SEEDS}, 'done': f'{nd}/{len(SEEDS)}'})
print('\n· = 대기, … = 진행중(추정), 숫자 = 완료 SR%')

with open(OUT / f'progress_grid_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['model'] + [f'seed{s}' for s in SEEDS] + ['done'])
    w.writeheader(); w.writerows(grid_rows)
print('saved:', OUT / f'progress_grid_{STAMP}.csv')

## 2b) 격자 이미지(PNG)


In [ ]:
# ── 격자 PNG (팀에 이미지로) ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(1.6 + 1.1 * len(SEEDS), 0.6 + 0.5 * len(MODELS)))
ax.axis('off')
col_lbl = [f'seed{s}' for s in SEEDS] + ['done']
cells, colours = [], []
for m in MODELS:
    row, crow = [], []
    for s in SEEDS:
        if (m, s) in done:
            sr = done[(m, s)]['sr']
            row.append(f'{sr:.1f}' if sr is not None else '?')
            crow.append('#c7e9c0' if sr is not None else '#eeeeee')
        elif (m, s) in running:
            row.append('…'); crow.append('#fff3bf')
        else:
            row.append('·'); crow.append('#f5f5f5')
    nd = sum((m, s) in done for s in SEEDS)
    row.append(f'{nd}/{len(SEEDS)}'); crow.append('#ffffff')
    cells.append(row); colours.append(crow)
t = ax.table(cellText=cells, rowLabels=MODELS, colLabels=col_lbl,
             cellColours=colours, loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1, 1.5)
ax.set_title(f'LIBERO-10 eval progress  {n_done}/{len(EXPECTED)}   ({STAMP})', fontsize=12, pad=10)
png = OUT / f'progress_grid_{STAMP}.png'
fig.savefig(png, dpi=150, bbox_inches='tight'); plt.close(fig)
print('saved:', png)

## 3) 지금까지 모델별 SR (완료 seed 만)


In [ ]:
# ── 지금까지 모델별 SR (완료된 seed 만, mean±std) ──
print(f"{'model':<14}{'SR mean':>9}{'± std':>8}{'seeds':>7}")
print('-' * 38)
sum_rows = []
for m in MODELS:
    vals = [done[(m, s)]['sr'] for s in SEEDS if (m, s) in done and done[(m, s)]['sr'] is not None]
    if vals:
        mean, std = float(np.mean(vals)), (float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0)
        print(f'{m:<14}{mean:>9.2f}{std:>8.2f}{len(vals):>5}/{len(SEEDS)}')
        sum_rows.append({'model': m, 'SR_mean': round(mean, 2), 'std': round(std, 2),
                         'n_seed': f'{len(vals)}/{len(SEEDS)}'})
    else:
        print(f'{m:<14}{"-":>9}{"-":>8}{0:>5}/{len(SEEDS)}')
with open(OUT / f'summary_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['model', 'SR_mean', 'std', 'n_seed'])
    w.writeheader(); w.writerows(sum_rows)
print('\nsaved:', OUT / f'summary_{STAMP}.csv')

# 500ep 미만 경고
under = [(m, s, done[(m, s)]['n_ep']) for (m, s) in done
         if done[(m, s)]['n_ep'] is not None and done[(m, s)]['n_ep'] < TARGET_EP]
if under:
    print(f'\n⚠️ {TARGET_EP}ep 미만 run {len(under)}개:', [(m, s, n) for m, s, n in under])

## 4) 떨림 — jerk RMS · LDJ · SPARC · Sign Flip (완료 + action 있는 것)


In [ ]:
# ── 떨림 지표 (완료 + action.pt 있는 것): jerk RMS · LDJ · SPARC · Sign Flip ──
trajs = {}
for (m, s), d in done.items():
    if not d['has_actions']:
        continue
    tr = cf.v23._load_action_trajs(ROOT / d['path'] / 'actions') or []
    if tr:
        trajs.setdefault(m, []).extend(tr)

if not trajs:
    print('action(.pt) 아직 없음 — SR/진행률만 (완료된 eval 이 늘면 다시 실행)')
else:
    import smooth_metrics as sm
    print(f"{'model':<14}{'jerk_RMS':>10}{'LDJ':>10}{'SPARC':>10}{'sign_flip':>11}{'n_ep':>6}")
    print('  (smoother =    lower      higher     near-0        lower)')
    print('-' * 61)
    srows = []
    for m in MODELS:
        if m not in trajs:
            continue
        agg = sm.aggregate_smoothness(trajs[m], chunk_size=100, fs=cf.fps_of(TASK))
        srows.append({'model': m, 'jerk_rms': round(agg['jerk_rms_mean'], 5),
                      'LDJ': round(agg['ldj_mean'], 3), 'SPARC': round(agg['sparc_mean'], 3),
                      'sign_flip': round(agg['sign_flips_mean'], 4), 'n_ep': len(trajs[m])})
        print(f"{m:<14}{agg['jerk_rms_mean']:>10.5f}{agg['ldj_mean']:>10.3f}"
              f"{agg['sparc_mean']:>10.3f}{agg['sign_flips_mean']:>11.4f}{len(trajs[m]):>6}")
    with open(OUT / f'smoothness_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['model', 'jerk_rms', 'LDJ', 'SPARC', 'sign_flip', 'n_ep'])
        w.writeheader(); w.writerows(srows)
    print('\nsaved:', OUT / f'smoothness_{STAMP}.csv')

## 5) zip → 팀에 이 파일 하나


In [ ]:
# ── 요약 MD + 전부 zip → 팀에 이 파일 하나 ──
lines = [f'# LIBERO-10 eval 중간 결과 ({STAMP})', '',
         f'- 진행: **{n_done}/{len(EXPECTED)}** 완료 (모델 {MODELS}, seed {SEEDS})',
         f'- 완료 eval 은 150k 체크포인트 × {TARGET_EP}ep', '',
         '자세한 표: progress_grid / summary / smoothness CSV·PNG 참고.']
(OUT / f'README_{STAMP}.md').write_text('\n'.join(lines), encoding='utf-8')

import shutil
zip_path = shutil.make_archive(str(cf.OUTPUT_BASE / 'share' / f'{TASK}_progress_{STAMP}'), 'zip', root_dir=OUT)
print('보낼 파일:', zip_path, '\n')
for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.0f} KB)')